# SpaPath multiple myeloma workflows

This notebook compares four reference configurations for the hMM2 disease section: scATAC alone, scRNA alone, ST alone, and all three references together. Each workflow ends after constructing the all-gene disease dataset.

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import scanpy as sc

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import spapath_model
import spapath_utils

warnings.filterwarnings("ignore")

In [ ]:
DATASET_ID = "MM"
DISEASE_SECTION = "hMM2"
DEVICE = "cuda"
SEED = 123

DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_ROOT / DATASET_ID
OUTPUT_DIR = PROJECT_ROOT / "outputs" / DATASET_ID
FIGURE_DIR = OUTPUT_DIR / "fig"

for output_path in (OUTPUT_DIR, FIGURE_DIR):
    spapath_utils.create_dir(output_path)

REFERENCE_CONFIGS = {
    "scATAC": {"hHBM_scATAC": "scATAC"},
    "scRNA": {"hHBM_scRNA": "sc"},
    "ST": {"hHBM1_ST": "ST"},
    "combined": {
        "hHBM_scATAC": "scATAC",
        "hHBM_scRNA": "sc",
        "hHBM1_ST": "ST",
    },
}

## 1. Shared workflow

In [ ]:
def run_reference_workflow(reference_type_map, run_name):
    adata_type_map = {
        **reference_type_map,
        DISEASE_SECTION: "ST",
    }
    sections = list(adata_type_map)

    reference_adatas = [
        sc.read_h5ad(PROCESSED_DIR / f"{section}.h5ad")
        for section in reference_type_map
    ]
    disease_adata = sc.read_h5ad(
        PROCESSED_DIR / f"{DISEASE_SECTION}.h5ad"
    )

    if "image_embedding" not in disease_adata.obsm:
        raise KeyError(
            f"{DISEASE_SECTION} is missing .obsm['image_embedding']."
        )
    image_embedding_dim = disease_adata.obsm["image_embedding"].shape[1]
    for reference_adata in reference_adatas:
        if "image_embedding" not in reference_adata.obsm:
            reference_adata.obsm["image_embedding"] = np.zeros(
                (reference_adata.n_obs, image_embedding_dim),
                dtype=np.float32,
            )

    batch_list = [*reference_adatas, disease_adata]
    adata_full, disease_adata_all_genes = spapath_utils.preprocess(
        adata_list=batch_list,
        adata_type_map=adata_type_map,
        full_num_hvgs=3000,
        min_genes_qc=10,
        min_cells_qc=10,
    )
    adata_full = spapath_utils.build_graph_GAT_plus(
        adata_full=adata_full,
        adata_type_map=adata_type_map,
        K=8,
        img_threshold=0.0,
    )

    model = spapath_model.Model(
        adata_full=adata_full,
        adata_type_map=adata_type_map,
        lr_pre=1e-4,
        lr=1e-4,
        n_pre_training_steps=500,
        n_training_steps=300,
        device=DEVICE,
        seed=SEED,
    )
    adata_full = model.initial_embedding()
    adata_full = model.clustering(
        init_res=1.5,
        intopk=40,
    )
    adata_full = model.integrate(topk=40)

    adata_full = spapath_utils.detection(
        adata=adata_full,
        embed="cell_embed",
        section_ids=sections,
        label_core="Pathological regions",
        label_other="Healthy-like regions",
        core_types=None,
        celltype_key=None,
        batch_key="batch",
        seed=SEED,
        neighbors=30,
        threshold=0.05,
        strategy="individual",
    )

    spapath_utils.plot_detection_umap(
        adata=adata_full,
        embed="cell_embed",
        section_id=DISEASE_SECTION,
        batch_key="batch",
        label_key="pred_label",
        seed=SEED,
        point_size=18,
        save=FIGURE_DIR / f"{DISEASE_SECTION}_{run_name}_detection_umap.png",
    )

    disease_data = spapath_utils.build_disease_data(
        adata_full=adata_full,
        disease_adata_all_genes=disease_adata_all_genes,
        disease_section=DISEASE_SECTION,
    )
    return adata_full, disease_data

## 2. scATAC reference

In [ ]:
scatac_adata_full, scatac_disease_data = run_reference_workflow(
    reference_type_map=REFERENCE_CONFIGS["scATAC"],
    run_name="scATAC",
)

## 3. scRNA reference

In [ ]:
scrna_adata_full, scrna_disease_data = run_reference_workflow(
    reference_type_map=REFERENCE_CONFIGS["scRNA"],
    run_name="scRNA",
)

## 4. ST reference

In [ ]:
st_adata_full, st_disease_data = run_reference_workflow(
    reference_type_map=REFERENCE_CONFIGS["ST"],
    run_name="ST",
)

## 5. Combined scATAC, scRNA, and ST references

In [ ]:
combined_adata_full, combined_disease_data = run_reference_workflow(
    reference_type_map=REFERENCE_CONFIGS["combined"],
    run_name="combined",
)